# Capa Silver : Dimensiones (advertisers, campaigns, ads)

**Qué hace esta notebook:** agarra las 3 tablas **dimensiones** de bronze y las deja limpias en el nuevo schema `dev.silver_ads`, una tabla silver por cada bronze.

**Por qué solo 3 tablas acá:** las otras 3 (impressions, clicks, conversions) son *eventos* y se transforman en la notebook `04_silver_eventos`, porque comparten otra lógica (timestamps, duplicados de eventos, registros huérfanos).

**Limpieza que aplica:**

| Problema en bronze | Dónde | Solución en silver |
|---|---|---|
| Filas duplicadas (se recarga el mismo archivo en cada ingesta) | las 3 | `QUALIFY row_number()` → 1 fila por `id` |
| Websites sin `https://` (~2%) | advertisers | se les agrega el protocolo |
| Presupuesto diario en $0 (~1%, valor inválido) | campaigns | pasa a `NULL` (dato faltante, no se borra la fila) |
| `creative_url` nulo (~1%) | ads | se deja `NULL` (inventar una URL sería mentir los datos) |
| Textos con espacios de más | las 3 | `trim()` |

**Idempotencia:** todas las escrituras son `MERGE` por `id`: re-ejecutar esta notebook N veces deja la tabla idéntica (actualiza lo existente, inserta solo lo nuevo).

**Por qué tablas creadas con schema explícito:** a diferencia de bronze (que COPY INTO infiere solo), silver declara sus columnas y tipos. Así el contrato de la capa queda documentado y un cambio inesperado en bronze sale a la luz

In [0]:
%sql
-- Paso 0: crear el SCHEMA silver una sola vez.
-- IF NOT EXISTS lo hace seguro de re-ejecutar (idempotente a nivel DDL).
CREATE SCHEMA IF NOT EXISTS dev.silver_ads
COMMENT 'Capa silver del dominio ads: tablas limpias, deduplicadas y normalizadas';

In [0]:
%sql
-- ADVERTISERS: la dimensión "quiénes invierten".
-- Problemas que trae bronze:
--  1) DUPLICADOS: cada corrida de la ingesta vuelve a aterrizar el mismo archivo (el generador
--     produce siempre los mismos 200 advertisers) y COPY INTO lo carga de nuevo → 600 filas por 200 reales.
--     Solución: 1 fila por id con QUALIFY row_number() (función de ventana).
--  2) FORMATO: ~2% de websites sin "https://" → se les agrega el protocolo.
--  3) NULOS: un website vacío pasa a NULL (se distingue de un valor válido).
--  4) STRINGS: trim() para sacar espacios de más.
--
-- IDEMPOTENCIA: MERGE por id → correrlo 10 veces deja la tabla igual.

CREATE TABLE IF NOT EXISTS dev.silver_ads.advertisers (
  id              BIGINT  NOT NULL COMMENT 'Id del advertiser',
  advertiser_name STRING           COMMENT 'Nombre de la empresa',
  industry        STRING           COMMENT 'Industria',
  website         STRING           COMMENT 'Sitio web normalizado (siempre con https://)',
  country         STRING           COMMENT 'País de origen'
) COMMENT 'advertisers limpio: deduplicado, websites normalizados';

MERGE INTO dev.silver_ads.advertisers AS t
USING (
  SELECT
    id,
    trim(advertiser_name) AS advertiser_name,
    trim(industry)        AS industry,
    CASE
      WHEN website IS NULL OR trim(website) = '' THEN NULL                        -- nulo real
      WHEN trim(website) NOT LIKE 'http%' THEN concat('https://', trim(website))    -- sin protocolo
      ELSE trim(website)
    END                   AS website,
    trim(country)         AS country
  FROM dev.bronze_ads.bronze_advertisers
  QUALIFY row_number() OVER (PARTITION BY id ORDER BY id) = 1                      -- 1 fila por id
) AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
200,200,0,0


In [0]:
%sql
-- CAMPAIGNS: la dimensión "cuánto y dónde invierte cada advertiser".
-- Problemas que trae bronze:
--  1) DUPLICADOS: 6.000 filas por 2.000 campañas reales (mismo motivo que advertisers) → dedup por id.
--  2) VALOR INVÁLIDO → NULL: ~1% de campañas con daily_budget_usd = 0. Un presupuesto de $0 no existe
--     como dato real: se interpreta como faltante y pasa a NULL. No se borra la fila ni se inventa valor.
--  3) STRINGS: trim() en los textos.

CREATE TABLE IF NOT EXISTS dev.silver_ads.campaigns (
  id               BIGINT  NOT NULL COMMENT 'Id de la campaña',
  advertiser_id    BIGINT           COMMENT 'FK a silver.advertisers',
  campaign_name   STRING           COMMENT 'Nombre de la campaña',
  channel         STRING           COMMENT 'Canal (Search, Social, Display, Video)',
  objective       STRING           COMMENT 'Objetivo (Awareness, Traffic, Conversion)',
  daily_budget_usd DOUBLE          COMMENT 'Presupuesto diario USD (NULL = dato faltante, era 0)'
) COMMENT 'campaigns limpio: deduplicado, presupuesto 0 normalizado a NULL';

MERGE INTO dev.silver_ads.campaigns AS t
USING (
  SELECT
    id,
    advertiser_id,
    trim(campaign_name) AS campaign_name,
    trim(channel)        AS channel,
    trim(objective)     AS objective,
    CASE WHEN daily_budget_usd = 0 THEN NULL ELSE daily_budget_usd END AS daily_budget_usd
  FROM dev.bronze_ads.bronze_campaigns
  QUALIFY row_number() OVER (PARTITION BY id ORDER BY id) = 1
) AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2000,2000,0,0


In [0]:
%sql
-- ADS: la dimensión "qué creatividad se muestra". Es la más grande (500 mil reales).
-- Problemas que trae bronze:
--  1) DUPLICADOS: 1.500.000 filas por 500.000 ads reales → dedup por id.
--  2) NULOS: ~1% de anuncios sin creative_url. DECISIÓN: se deja NULL a propósito.
--     Inventar una URL falsa sería peor (mentiría sobre los datos). Si en gold hace falta
--     saber si tiene creatividad, se deriva la columna has_creative ahí.

CREATE TABLE IF NOT EXISTS dev.silver_ads.ads (
  id           BIGINT NOT NULL COMMENT 'Id del anuncio',
  campaign_id  BIGINT          COMMENT 'FK a silver.campaigns',
  format       STRING          COMMENT 'Formato (Banner, Video, Native, SearchText)',
  creative_url STRING          COMMENT 'URL de la creatividad (NULL = sin creatividad)'
) COMMENT 'ads limpio: deduplicado, nulos documentados';

MERGE INTO dev.silver_ads.ads AS t
USING (
  SELECT
    id,
    campaign_id,
    trim(format)      AS format,
    trim(creative_url) AS creative_url
  FROM dev.bronze_ads.bronze_ads
  QUALIFY row_number() OVER (PARTITION BY id ORDER BY id) = 1
) AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
500000,500000,0,0


In [0]:
%sql
-- VERIFICACIÓN: comprobar que silver quedó como esperamos.
-- Chequeo 1 → duplicados: filas tiene que ser IGUAL a ids_distintos.
--   Valores esperados: advertisers 200, campaigns 2.000, ads 500.000.
SELECT 'advertisers' AS tabla, COUNT(*) AS filas, COUNT(DISTINCT id) AS ids_distintos
FROM dev.silver_ads.advertisers
UNION ALL
SELECT 'campaigns', COUNT(*), COUNT(DISTINCT id) FROM dev.silver_ads.campaigns
UNION ALL
SELECT 'ads', COUNT(*), COUNT(DISTINCT id) FROM dev.silver_ads.ads;

tabla,filas,ids_distintos
advertisers,200,200
campaigns,2000,2000
ads,500000,500000


In [0]:
%sql
-- Chequeo 2 → reglas de limpieza:
--   webs_sin_https   = 0    (todos los websites válidos quedaron con protocolo)
--   budgets_null     = 31   (los presupuestos 0 pasaron a NULL, no desaparecieron)
--   creatives_null   ≈ 4.844 (1% de 500.000: se mantienen como NULL documentados)
SELECT
  (SELECT COUNT(*) FROM dev.silver_ads.advertisers
    WHERE website IS NOT NULL AND website NOT LIKE 'https://%') AS webs_sin_https,
  (SELECT COUNT(*) FROM dev.silver_ads.campaigns
    WHERE daily_budget_usd IS NULL)                                AS budgets_null,
  (SELECT COUNT(*) FROM dev.silver_ads.ads
    WHERE creative_url IS NULL)                                   AS creatives_null;

webs_sin_https,budgets_null,creatives_null
0,31,4844
